# **A Macro Simulation**

> **Objective:** to build a simple, data-driven framework to interpret the current macroeconomic and credit cycle using U.S. macroeconomic data.  

My goal is not to forecast GDP or to estimate a full-scale macro model (e.g., DSGE).
Instead, I want to **translate a qualitative narrative into a minimal dynamic system** and show how the interaction between:

- **transactions** (income $\rightarrow$ spending),
- **credit creation** and **debt accumulation**,
- **interest rates** and **policy responses**,

can generate **short-term business cycles** and **long-term debt cycles** in a stylized way.

### **References**
- Blanchard, O., Amighini, A., & Giavazzi, F. (2021), *Macroeconomia. Una prospettiva europea* (8th ed.). Pearson Education.
- Ray Dalio (2013), *[How the Economic Machine Works](https://www.youtube.com/watch?v=PHe0bXAIuk0)*

### **Contents**
- [Setup](#setup)
- [Data from FRED (download, cleaning, transformations)](#data-download-and-preprocessing)
- [Stylized facts (macro series and relationships)](#stylized-facts)
- [Credit equation calibration (toy calibration)](#credit-equation-calibration)
- [Credit Cycle Index (HP gaps + robust PCA)](#credit-cycle-index)
- [Cycle clock and regime diagnostics](#cycle-clock)


### **Setup**

I keep the setup minimal and reproducible: fixed seed, API key from environment variables, and standard scientific libraries.


In [1]:
# --------------------------------------------------
# Setup
# --------------------------------------------------
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import os
from fredapi import Fred
from dotenv import load_dotenv
load_dotenv()
import statsmodels.api as sm
from statsmodels.tsa.filters.hp_filter import hpfilter

# ----------------------------
# Reproducibility
# ----------------------------
# I fix the random seed so that any stochastic component is reproducible.
SEED = 1
np.random.seed(SEED)

# ----------------------------
# FRED API key handling
# ----------------------------
# I do NOT hard-code the API key in the notebook.
# I read it from an environment variable to avoid leaking credentials on GitHub.
FRED_API_KEY = os.getenv("FRED_API_KEY")

if FRED_API_KEY is None:
    raise RuntimeError("FRED_API_KEY not found. Please set it as an environment variable before running this notebook.")

fred = Fred(api_key=FRED_API_KEY)
print("FRED API connection successfully initialized.")


FRED API connection successfully initialized.


### **Data download and preprocessing**

I download a small set of U.S. macroeconomic time series from FRED, each capturing a key dimension of the business and credit cycle: real activity, inflation, monetary policy, labor market conditions, and bank credit.

1. [Real GDP](https://fred.stlouisfed.org/series/GDP#) (quarterly): `GDPC1`
2. [CPI](https://fred.stlouisfed.org/series/CPIAUCSL) (monthly): `CPIAUCSL`
3. [Fed Funds Rate](https://fred.stlouisfed.org/series/FEDFUNDS) (monthly): `FEDFUNDS`
4. [Unemployment rate](https://fred.stlouisfed.org/series/UNRATE) (monthly): `UNRATE`
5. [Bank credit proxy](https://fred.stlouisfed.org/series/TOTLL) (weekly): `TOTLL`

I want a coherent frequency for the model and the cycle diagnostics.
GDP is naturally quarterly, and Dalio-style narratives are typically discussed in “quarters” rather than “months”.
So I aggregate all monthly/weekly series to **quarterly frequency**.

In [2]:
# --------------------------------------------------
# Series configuration
# --------------------------------------------------
SERIES = {
    "GDP": "GDPC1",        # quarterly
    "CPI": "CPIAUCSL",     # monthly
    "FFR": "FEDFUNDS",     # monthly
    "UNRATE": "UNRATE"     # monthly
}

CREDIT_SERIES = "TOTLL"   # weekly bank credit proxy
START = "1950-01-01"
END = None # up to the most recent available date

In [3]:
# --------------------------------------------------
# Download helpers
# --------------------------------------------------
def fetch_series(series_id: str, start: str, end=None) -> pd.Series:
    s = fred.get_series(series_id, observation_start=start, observation_end=end)
    s = pd.Series(s)
    s.index = pd.to_datetime(s.index)
    return s.sort_index()

def download_macro(series_dict: dict, start: str, end=None) -> pd.DataFrame:
    out = {}
    for name, sid in series_dict.items():
        print(f"Downloaded {name} ({sid})")
        out[name] = fetch_series(sid, start, end)
    return pd.concat(out.values(), axis=1).set_axis(out.keys(), axis=1)

In [4]:
# --------------------------------------------------
# Download raw data
# --------------------------------------------------
macro_raw = download_macro(SERIES, start=START, end=END)

totll_w = fetch_series(CREDIT_SERIES, start=START, end=END)
totll_w.name = "TOTLL"

Downloaded GDP (GDPC1)
Downloaded CPI (CPIAUCSL)
Downloaded FFR (FEDFUNDS)
Downloaded UNRATE (UNRATE)


In [5]:
# --------------------------------------------------
# Quarterly panel construction
# --------------------------------------------------
def to_quarterly_panel(macro_df: pd.DataFrame, credit_weekly: pd.Series) -> pd.DataFrame:
    """
    I build a quarterly dataset with consistent timing:
    - GDP is already quarterly -> I take last observation in the quarter (usually already quarter-end).
    - CPI is monthly -> I take quarterly average (inflation is a flow-like measure).
    - FFR is monthly -> I take quarterly average (policy stance across the quarter).
    - UNRATE is monthly -> I take quarterly average (labor market condition across the quarter).
    - TOTLL is weekly -> I take the end-of-quarter level (stock measure).
    """
    df = macro_df.copy()

    # GDP (quarterly): keep as quarter-end series
    gdp_q = df["GDP"].resample("QE").last()

    # CPI (monthly -> quarterly average)
    cpi_q = df["CPI"].resample("QE").mean()

    # FFR (monthly -> quarterly average)
    ffr_q = df["FFR"].resample("QE").mean()

    # UNRATE (monthly -> quarterly average)
    un_q = df["UNRATE"].resample("QE").mean()

    # Credit (weekly -> quarter-end)
    credit_q = credit_weekly.resample("QE").last()

    out = pd.concat(
        [gdp_q.rename("GDP"),
         cpi_q.rename("CPI"),
         ffr_q.rename("FFR"),
         un_q.rename("UNRATE"),
         credit_q.rename("TOTLL")],
        axis=1
    ).dropna()

    return out

qdata_raw = to_quarterly_panel(macro_raw, totll_w)

In [6]:
# --------------------------------------------------
# Transformations (quarterly)
# --------------------------------------------------
def add_transformations_quarterly(df_q: pd.DataFrame) -> pd.DataFrame:
    """
    I compute a few standard transformations at quarterly frequency:
    - YoY GDP growth (shift 4)
    - YoY CPI inflation (shift 4)
    - YoY change in unemployment rate (level difference over 4 quarters)
    - YoY credit growth (shift 4)
    - QoQ log credit growth (for calibration)
    - Ex-post real policy rate proxy: FFR - CPI_YoY
    """
    df = df_q.copy()

    # YoY rates (quarterly: 4-quarter shift)
    df["GDP_YoY"] = 100 * (df["GDP"] / df["GDP"].shift(4) - 1)
    df["CPI_YoY"] = 100 * (df["CPI"] / df["CPI"].shift(4) - 1)

    # Unemployment change over 1 year (percentage points)
    df["UNRATE_YoY_Delta"] = df["UNRATE"] - df["UNRATE"].shift(4)

    # Credit growth proxies
    df["TOTLL_YoY"] = 100 * (df["TOTLL"] / df["TOTLL"].shift(4) - 1)
    df["dC_qoq"] = np.log(df["TOTLL"]).diff()  # QoQ log growth

    # Real rate proxy (ex-post)
    df["Real_FFR"] = df["FFR"] - df["CPI_YoY"]

    # Clean sample (needs YoY rates)
    df = df.dropna(subset=["GDP_YoY", "CPI_YoY", "FFR", "UNRATE", "TOTLL", "TOTLL_YoY", "Real_FFR"]).copy()
    return df

qdata = add_transformations_quarterly(qdata_raw)

In [7]:
display(qdata.head())
display(qdata.tail())
display(qdata.describe().round(2))

,GDP,CPI,FFR,UNRATE,TOTLL,GDP_YoY,CPI_YoY,UNRATE_YoY_Delta,TOTLL_YoY,dC_qoq,Real_FFR
1974-03-31,6097.258,47.300000,9.323333,5.133333,491.0192,0.638998,9.914795,0.200000,14.743924,0.025794,-0.591461
1974-06-30,6111.751,48.566667,11.250000,5.200000,515.5043,-0.208262,10.546282,0.266667,15.089863,0.048662,0.703718
1974-09-30,6053.978,49.933333,12.090000,5.633333,525.3017,-0.629040,11.458333,0.833333,13.488421,0.018827,0.631667
1974-12-31,6030.464,51.466667,9.346667,6.600000,531.2668,-1.945763,12.046444,1.833333,11.023878,0.011292,-2.699777
1975-03-31,5957.035,52.566667,6.303333,8.266667,521.5394,-2.299771,11.134602,3.133333,6.215684,-0.018480,-4.831268


,GDP,CPI,FFR,UNRATE,TOTLL,GDP_YoY,CPI_YoY,UNRATE_YoY_Delta,TOTLL_YoY,dC_qoq,Real_FFR
2024-09-30,23478.570,314.182667,5.263333,4.166667,12497.9390,2.791390,2.659872,0.500000,2.340452,0.005617,2.603462
2024-12-31,23586.542,316.538667,4.650000,4.133333,12605.7539,2.399788,2.719489,0.333333,2.811055,0.008590,1.930511
2025-03-31,23548.210,319.492000,4.330000,4.100000,12744.7556,2.019273,2.739026,0.266667,3.163478,0.010966,1.590974
2025-06-30,23770.976,320.800333,4.330000,4.166667,12921.6119,2.080467,2.460803,0.166667,3.972319,0.013781,1.869197
2025-09-30,24024.957,323.288000,4.293333,4.300000,13098.6470,2.327173,2.898102,0.133333,4.806456,0.013608,1.395232


,GDP,CPI,FFR,UNRATE,TOTLL,GDP_YoY,CPI_YoY,UNRATE_YoY_Delta,TOTLL_YoY,dC_qoq,Real_FFR
count,207.00,207.00,207.00,207.00,207.00,207.00,207.00,207.00,207.00,207.00,207.00
mean,13656.77,171.85,4.80,6.12,4594.92,2.68,3.93,-0.01,6.77,0.02,0.87
std,5188.92,71.85,3.92,1.75,3630.45,2.27,2.96,1.45,4.46,0.01,2.82
min,5957.04,47.30,0.06,3.53,491.02,-7.40,-1.61,-7.07,-6.70,-0.04,-7.90
25%,8953.28,111.13,1.25,4.78,1584.08,1.74,2.02,-0.62,4.11,0.01,-1.01
50%,13827.98,168.43,4.87,5.83,3373.18,2.79,3.07,-0.27,7.02,0.02,0.57
75%,17479.75,230.61,6.66,7.30,6896.89,4.10,4.65,0.33,9.53,0.02,2.87
max,24024.96,323.29,17.78,13.00,13098.65,12.39,14.43,9.37,17.41,0.06,7.91


In [8]:
print(qdata.index.min(), qdata.index.max(), qdata.shape)

1974-03-31 00:00:00 2025-09-30 00:00:00 (207, 11)


In [9]:
# --------------------------------------------------
# Export cleaned data (quarterly)
# --------------------------------------------------
os.makedirs("data", exist_ok=True)
qdata.to_csv("data/data_quarterly.csv", index_label="Date")

### **Stylized facts**

I plot a few simple relationships to check if the series behave as expected.

In [10]:
# --------------------------------------------------
# Stylized facts: GDP growth vs inflation
# --------------------------------------------------
fig = go.Figure()
fig.add_trace(go.Scatter(x=qdata.index, y=qdata["GDP_YoY"], mode="lines", name="GDP YoY (%)"))
fig.add_trace(go.Scatter(x=qdata.index, y=qdata["CPI_YoY"], mode="lines", name="CPI YoY (%)"))

fig.update_layout(
    title="GDP Growth vs Inflation (QE, YoY, %)",
    xaxis_title="Date",
    yaxis_title="Percent",
    hovermode="x unified",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig.show()

GDP growth exhibits higher short-term volatility and sharp contractions during recessionary episodes, most notably around the Global Financial Crisis and the COVID-19 shock. Inflation, by contrast, displays smoother dynamics, with elevated and persistent levels during the 1970s and more stable behavior from the mid-1990s onward, aside from recent post-pandemic pressures.
Overall, the figure highlights the cyclical nature of real activity relative to the more inertial behavior of inflation, as well as the strong comovement between the two variables during major macroeconomic shocks.  

The limited comovement between GDP growth and inflation observed in the figure is economically expected. Real activity responds rapidly to cyclical and financial shocks, while inflation exhibits greater persistence and typically adjusts with significant lags due to nominal rigidities, wage-setting mechanisms, and inflation expectations. As a result, business cycle fluctuations and inflation dynamics often follow distinct and partially asynchronous paths rather than moving in tandem.

In [11]:
# --------------------------------------------------
# Stylized facts: FFR vs inflation
# --------------------------------------------------
fig = go.Figure()
fig.add_trace(go.Scatter(x=qdata.index, y=qdata["CPI_YoY"], mode="lines", name="CPI YoY (%)"))
fig.add_trace(go.Scatter(x=qdata.index, y=qdata["FFR"], mode="lines", name="Fed Funds Rate (%)"))

fig.add_hline(y=0, line_dash="dot", line_width=1, opacity=0.5)

fig.update_layout(
    title="Monetary Policy vs Inflation (QE, FFR and CPI YoY, %)",
    xaxis_title="Date",
    yaxis_title="Percent",
    hovermode="x unified",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig.show()

The figure illustrates the relationship between inflation and the monetary policy stance over time. The Federal Funds Rate broadly reacts to movements in inflation, tightening during periods of elevated price pressures and easing when inflation declines or economic conditions weaken. However, the adjustment is not mechanical: policy responses occur with delays and are influenced by expectations, financial stability concerns, and the broader macroeconomic outlook. This generates episodes in which inflation and the policy rate temporarily diverge, especially around recessions and major shocks.

In [12]:
# --------------------------------------------------
# Stylized facts: GDP growth vs unemployment (dual axis)
# --------------------------------------------------
fig = go.Figure()
fig.add_trace(go.Scatter(x=qdata.index, y=qdata["GDP_YoY"], mode="lines", name="GDP YoY Growth (%)"))
fig.add_trace(go.Scatter(x=qdata.index, y=qdata["UNRATE"], mode="lines", name="Unemployment Rate (%)", yaxis="y2"))

fig.update_layout(
    title="Business Cycle (QE, GDP Growth vs Unemployment)",
    xaxis_title="Date",
    yaxis_title="GDP Growth (%)",
    yaxis2=dict(title="Unemployment Rate (%)", overlaying="y", side="right"),
    hovermode="x unified",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig.show()

The figure highlights the inverse relationship between real economic activity and labor market conditions over the business cycle. Periods of strong GDP growth are typically associated with declining unemployment, while economic slowdowns and recessions coincide with rising unemployment rates. The adjustment in unemployment lags changes in output, reflecting labor market frictions and firms’ cautious hiring and firing decisions. Sharp downturns, such as during major crises, generate abrupt contractions in output followed by pronounced increases in unemployment, clearly illustrating the cyclical nature of the economy.

In [13]:
# --------------------------------------------------
# Stylized facts: Credit growth (TOTLL YoY)
# --------------------------------------------------
fig = go.Figure()
fig.add_trace(go.Scatter(x=qdata.index, y=qdata["TOTLL_YoY"], mode="lines", name="TOTLL YoY (%)"))

fig.update_layout(
    title="Bank Credit Growth (QE, TOTLL, YoY %)",
    xaxis_title="Date",
    yaxis_title="Percent",
    hovermode="x unified",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig.show()

The figure shows the year-over-year growth rate of bank credit, capturing the cyclical behavior of credit creation over time. Credit growth tends to accelerate during economic expansions, supporting higher spending and investment, and to decelerate sharply during downturns, often turning negative in periods of financial stress. The pronounced contractions around major crisis episodes highlight the pro-cyclical nature of bank lending and its close interaction with the business cycle. Overall, the series illustrates how credit conditions amplify economic fluctuations rather than simply mirroring real activity.

### **Credit equation calibration**

In [14]:
# --------------------------------------------------
# Calibration: credit growth equation (quarterly)
# --------------------------------------------------
df_cal = qdata.copy()

# Dependent variable: QoQ log growth in credit
df_cal["dC"] = np.log(df_cal["TOTLL"]).diff()

# Regressors
df_cal["CREDIT_lag"] = np.log(df_cal["TOTLL"]).shift(1)
df_cal["Y_proxy"] = df_cal["GDP_YoY"] / 100.0
df_cal["r"] = df_cal["FFR"]

df_cal = df_cal.dropna(subset=["dC", "Y_proxy", "CREDIT_lag", "r"]).copy()

X = sm.add_constant(df_cal[["Y_proxy", "CREDIT_lag", "r"]])
y = df_cal["dC"]

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                     dC   R-squared:                       0.241
Model:                            OLS   Adj. R-squared:                  0.230
Method:                 Least Squares   F-statistic:                     21.42
Date:                Mon, 05 Jan 2026   Prob (F-statistic):           4.35e-12
Time:                        15:55:38   Log-Likelihood:                 615.15
No. Observations:                 206   AIC:                            -1222.
Df Residuals:                     202   BIC:                            -1209.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0073      0.012      0.599      0.5

### Credit Growth Equation: OLS Results

I estimate a simple quarterly credit growth equation using **Ordinary Least Squares (OLS)**.  
The regression is intended as a **toy calibration**, not as a structural or causal model.  
Its purpose is to verify whether the data support the qualitative mechanisms assumed in the stylized “Economic Machine” framework.

#### Model specification
The estimated equation is:

$$
\Delta C_t = \alpha \, Y_t - \theta \, C_{t-1} - \beta \, r_t + \varepsilon_t
$$

where:
- $\Delta C_t$ is quarterly log growth of bank credit,
- $Y_t$ is a proxy for economic activity (GDP YoY growth),
- $C_{t-1}$ is the lagged level of credit,
- $r_t$ is the nominal policy interest rate,
- $\varepsilon_t$ is an error term.

#### Goodness of fit
- $R^2 = 0.24$: the model explains around 24% of the variation in quarterly credit growth.
- The **F-statistic is highly significant**, indicating that the regressors are jointly informative.
- This level of explanatory power is reasonable for a reduced-form macro-financial regression.

#### Coefficient interpretation
- **Income proxy ($Y_t$)**  
  The coefficient on GDP growth is positive and highly significant.  
  This suggests that credit creation is **procyclical**: stronger economic activity is associated with faster credit growth.

- **Lagged credit level ($C_{t-1}$)**  
  The coefficient is negative but statistically insignificant.  
  This indicates weak evidence, in this simple specification, of a direct debt overhang effect on short-run credit growth.

- **Policy rate ($r_t$)**  
  The coefficient on the nominal interest rate is statistically significant.  
  Its sign reflects the empirical correlation between policy tightening and credit dynamics in the sample, rather than a structural causal effect.

- **Intercept**  
  The constant term is small and not statistically significant, suggesting limited autonomous credit growth once macro conditions are controlled for.

#### Diagnostics and caveats
- The Durbin–Watson statistic indicates some persistence in residuals, which is typical in macroeconomic time series.
- Residuals deviate mildly from normality, as shown by the Jarque–Bera test.
- Standard errors are non-robust; inference should therefore be interpreted with caution.

#### Interpretation
Overall, the regression supports the **qualitative narrative** underlying the model:
credit growth tends to be procyclical and related to macroeconomic conditions.
However, the results should be read as **descriptive and illustrative**, not as a structural estimation of credit supply or demand.

In [15]:
# --------------------------------------------------
# Map regression coefficients to the model notation
# --------------------------------------------------
alpha_hat = model.params["Y_proxy"]
theta_hat = -model.params["CREDIT_lag"]  # I flip sign to match "- theta * D_{t-1}"
beta_hat  = -model.params["r"]           # I flip sign to match "- beta * r_t"

print("Calibrated parameters (toy mapping):")
print(f"alpha (income sensitivity): {alpha_hat:.6f}")
print(f"theta (credit level drag):  {theta_hat:.6f}")
print(f"beta (interest rate):       {beta_hat:.6f}")

Calibrated parameters (toy mapping):
alpha (income sensitivity): 0.237113
theta (credit level drag):  0.000277
beta (interest rate):       -0.000946


### **Credit Cycle Index**

I build an index based on three pillars:
1) Output conditions (GDP growth gap)
2) Credit conditions (credit growth gap)
3) Monetary stance (real policy rate, standardized)

#### Gaps via Hodrick-Prescott filter

Because this is quarterly, I use the standard HP lambda **1600**.

#### PCA sign convention

PCA is sign-indeterminate: the first component can flip without affecting the variance explained.  
To ensure interpretability, I adopt a consistent convention in which **higher values correspond to easier financial conditions**.  
Operationally, I compute the PCA-based score on standardized pillars and then **align its direction with a fixed-weight benchmark** (computed on the same standardized pillars).  
If the correlation between the two indices is negative, I flip the PCA score (and loadings) so that both measures move in the same economic direction.

In [16]:
# --------------------------------------------------
# Helper functions: z-score and HP gap (quarterly)
# --------------------------------------------------
# I compute a rolling z-score to standardize the real rate in a time-varying way
def zscore(x: pd.Series, window: int = 40, min_periods: int = 20) -> pd.Series:
    mu = x.rolling(window, min_periods=min_periods).mean()
    sd = x.rolling(window, min_periods=min_periods).std()
    return (x - mu) / sd

# I compute the HP cyclical component (two-sided)
def hp_gap(series: pd.Series, lamb: float = 1600.0) -> pd.Series:
    s = series.dropna()
    cyc, tr = hpfilter(s, lamb=lamb)
    out = pd.Series(index=series.index, dtype=float)
    out.loc[s.index] = cyc
    return out

def pca_pc1_svd(X_std: np.ndarray) -> np.ndarray:
    """
    Robust PCA PC1 via SVD on standardized data.
    - X_std must be T x K (already standardized).
    - Returns the loadings vector (K,).
    """
    # I use SVD because it is numerically stable and does not require me to build covariance matrices manually.
    # X_std = U * S * Vt, and PC1 loadings are the first row of Vt.
    U, S, Vt = np.linalg.svd(X_std, full_matrices=False)
    pc1 = Vt[0, :]  # first principal component loadings
    return pc1

### Principal Component Analysis (PCA)

Principal Component Analysis (PCA) is a statistical technique used to reduce the dimensionality of a multivariate dataset by transforming a set of correlated variables into a smaller number of **orthogonal components** that capture the maximum possible variance.

In this notebook, PCA is implemented **directly on the standardized data matrix** using **Singular Value Decomposition (SVD)**, as shown in the corresponding code cell.  
Let $X \in \mathbb{R}^{T \times K}$ denote the standardized return matrix, where each column has zero mean and unit variance. The SVD decomposition
$$
X = U \Sigma V^\top
$$
provides a numerically stable way to extract principal components without explicitly constructing the covariance matrix.

The **first principal component (PC1)** is obtained from the first row of $V^\top$ and represents the linear combination of assets that explains the largest share of common variation in returns. In a financial context, this component often captures the dominant *market-wide factor* driving co-movements across assets.

Using SVD-based PCA ensures robustness in high-dimensional settings and avoids potential numerical issues associated with eigenvalue decompositions of sample covariance matrices.

In [17]:
# Cycle diagnostics (quarterly):
# - HP gaps for GDP and credit (lambda=1600)
# - rolling z-score for real rates
# - composite index with fixed weights + PCA weights
# - regime classification

cycle = qdata.copy()

# 1) Pillars
cycle["Easing"] = -(zscore(cycle["Real_FFR"], window=40, min_periods=20))  # higher = easier stance

# 2) Gaps (HP)
cycle["GDP_gap_hp"]    = hp_gap(cycle["GDP_YoY"],   lamb=1600.0)
cycle["Credit_gap_hp"] = hp_gap(cycle["TOTLL_YoY"], lamb=1600.0)

cycle = cycle.dropna(subset=["GDP_gap_hp", "Credit_gap_hp", "Easing"]).copy()

# 3) Composite index: fixed weights (baseline, in original units)
cycle["CCI_fixed"] = (
    0.5 * cycle["Credit_gap_hp"]
    + 0.3 * cycle["GDP_gap_hp"]
    + 0.2 * cycle["Easing"]
)

# 4) Composite index: PCA weights (robust) — build on standardized pillars
X_pca = pd.DataFrame({
    "Credit_gap": cycle["Credit_gap_hp"],
    "GDP_gap": cycle["GDP_gap_hp"],
    "Easing": cycle["Easing"]
}).dropna()

# Standardize pillars (PCA on correlation)
X_std = ((X_pca - X_pca.mean()) / X_pca.std(ddof=0)).values

# Fixed weights on the SAME standardized pillars (fair comparison)
w_fixed = np.array([0.5, 0.3, 0.2])
cci_fixed_std = X_std @ w_fixed
cycle["CCI_fixed_std"] = pd.Series(cci_fixed_std, index=X_pca.index).reindex(cycle.index)

# PCA PC1 loadings
pc1 = pca_pc1_svd(X_std)

# Optional sign anchoring: keep Easing loading positive (interpretability)
if pc1[2] < 0:
    pc1 = -pc1

cci_pca = X_std @ pc1
cycle["CCI_pca"] = pd.Series(cci_pca, index=X_pca.index).reindex(cycle.index)

# CRITICAL FIX
# Ensure the PCA index has the same "direction" as the fixed index:
# if they are negatively correlated, flip PC1 and the PCA score.
corr_fp = cycle[["CCI_fixed_std", "CCI_pca"]].dropna().corr().iloc[0, 1]
if corr_fp < 0:
    pc1 = -pc1
    cycle["CCI_pca"] = -cycle["CCI_pca"]
    corr_fp = -corr_fp  # now positive by construction

print("PCA loadings (Credit_gap, GDP_gap, Easing):", np.round(pc1, 3))
print(f"Corr(CCI_fixed_std, CCI_pca): {corr_fp:.3f}")

# Choose which index I want downstream
CCI_NAME = "CCI_pca"  # or "CCI_fixed" or "CCI_fixed_std"
cycle["Credit_Cycle_Index"] = cycle[CCI_NAME]

# 5) Regime classification (4 quadrants)
def classify_regime(row) -> str:
    g = row["GDP_gap_hp"]
    c = row["Credit_gap_hp"]
    if (g >= 0) and (c >= 0):
        return "Expansion"
    if (g < 0) and (c >= 0):
        return "Late-cycle (slowdown)"
    if (g < 0) and (c < 0):
        return "Deleveraging"
    return "Recovery"

cycle["Regime"] = cycle.apply(classify_regime, axis=1)

# 6) Latest observation
latest = cycle.dropna(subset=["Credit_Cycle_Index"]).iloc[-1]

print("\n=== Current Cycle Diagnostics (Quarterly, Improved) ===")
print(f"{'Date:':30s} {latest.name.date()}")
print(f"{'GDP gap (HP):':30s} {latest['GDP_gap_hp']:>8.2f}")
print(f"{'Credit gap (HP):':30s} {latest['Credit_gap_hp']:>8.2f}")
print(f"{'Real policy rate:':30s} {latest['Real_FFR']:>8.2f}")
print(f"{f'Index ({CCI_NAME}, higher=easier):':30s} {latest['Credit_Cycle_Index']:>8.2f}")
print(f"{'Regime:':30s} {latest['Regime']}")

PCA loadings (Credit_gap, GDP_gap, Easing): [ 0.725  0.085 -0.683]
Corr(CCI_fixed_std, CCI_pca): 0.443

=== Current Cycle Diagnostics (Quarterly, Improved) ===
Date:                          2025-09-30
GDP gap (HP):                     -0.28
Credit gap (HP):                   1.08
Real policy rate:                  1.40
Index (CCI_pca, higher=easier):     0.81
Regime:                        Late-cycle (slowdown)


### **Plots: CCI (Fixed vs PCA)**

I compare the fixed-weight index with the PCA-based index.

In [18]:
# --------------------------------------------------
# Plot: Credit Cycle Index (fixed vs PCA)
# --------------------------------------------------
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=cycle.index, y=cycle["CCI_fixed"],
    mode="lines", name="CCI (fixed weights)", opacity=0.6
))

fig.add_trace(go.Scatter(
    x=cycle.index, y=cycle["CCI_pca"],
    mode="lines", name="CCI (PCA weights)"
))

fig.add_hline(y=0, line_dash="dash", opacity=0.6)

fig.update_traces(
    hovertemplate="Date=%{x|%Y-%m-%d}<br>Value=%{y:.2f}<extra></extra>"
)

fig.update_layout(
    title="Credit Cycle Index (Quarterly) — Fixed vs PCA",
    xaxis_title="Date",
    yaxis_title="Index (higher = easier conditions)",
    hovermode="x unified",
    template="plotly_white",
)

fig.show()

### **Quick checkpoints**

I compare the latest quarter with a few historical stress / cycle moments.

In [19]:
# --------------------------------------------------
# Quick comparison: today vs selected historical dates
# --------------------------------------------------
checkpoints = ["2006-12-31", "2009-03-31", "2019-12-31", "2020-06-30", str(cycle.index[-1].date())]

rows = []
for d in checkpoints:
    d = pd.to_datetime(d)
    nearest = cycle.index[cycle.index.get_indexer([d], method="nearest")[0]]
    r = cycle.loc[nearest, ["GDP_gap_hp", "Credit_gap_hp", "Real_FFR", "Credit_Cycle_Index", "Regime"]]
    rows.append([nearest.date(), *r.values])

summary = pd.DataFrame(rows, columns=["Date", "GDP_gap_hp", "Credit_gap_hp", "Real_FFR", "CCI", "Regime"])
summary

,Date,GDP_gap_hp,Credit_gap_hp,Real_FFR,CCI,Regime
0,2006-12-31,0.749123,3.396759,3.281271,1.539689,Expansion
1,2009-03-31,-4.041796,-1.337591,0.367570,-0.492747,Deleveraging
2,2019-12-31,1.108144,-0.701622,-0.405252,0.351645,Recovery
3,2020-06-30,-9.702378,3.302593,-0.349276,0.861940,Late-cycle (slowdown)
4,2025-09-30,-0.278264,1.077546,1.395232,0.808423,Late-cycle (slowdown)


### **Cycle clock**

I use the (GDP gap, Credit gap) plane as a “cycle clock”.
The four quadrants define the regime labels.

In [20]:
# --------------------------------------------------
# Cycle Clock: GDP gap vs Credit gap + highlight latest
# --------------------------------------------------
clock = cycle.dropna(subset=["GDP_gap_hp", "Credit_gap_hp", "Regime"]).copy()

today_date = clock.index[-1]
today = clock.loc[today_date]

fig = go.Figure()

# All history
fig.add_trace(go.Scatter(
    x=clock["GDP_gap_hp"],
    y=clock["Credit_gap_hp"],
    mode="markers",
    name="History",
    opacity=0.55,
    text=clock.index.strftime("%Y-%m-%d"),
    hovertemplate="Date=%{text}<br>GDP gap=%{x:.2f}<br>Credit gap=%{y:.2f}<extra></extra>"
))

# Highlight latest
fig.add_trace(go.Scatter(
    x=[today["GDP_gap_hp"]],
    y=[today["Credit_gap_hp"]],
    mode="markers+text",
    name="Latest quarter",
    text=[today_date.strftime("%Y-%m-%d")],
    textposition="top center",
    marker=dict(size=14),
    customdata=[today["Regime"]],
    hovertemplate="LATEST<br>Date=%{text}<br>GDP gap=%{x:.2f}<br>Credit gap=%{y:.2f}<br>Regime=%{customdata}<extra></extra>",
))

# Quadrant lines
fig.add_vline(x=0, line_dash="dash", opacity=0.5)
fig.add_hline(y=0, line_dash="dash", opacity=0.5)

fig.update_layout(
    title="Cycle Clock (Quarterly) — Output Gap vs Credit Gap",
    xaxis_title="GDP gap (HP-filtered)",
    yaxis_title="Credit gap (HP-filtered)",
    template="plotly_white",
)

fig.show()

print("=== Cycle Clock (Latest quarter) ===")
print(f"Date:   {today_date.date()}")
print(f"Regime: {today['Regime']}")
print(f"GDP gap (HP):    {today['GDP_gap_hp']:.2f}")
print(f"Credit gap (HP): {today['Credit_gap_hp']:.2f}")

=== Cycle Clock (Latest quarter) ===
Date:   2025-09-30
Regime: Late-cycle (slowdown)
GDP gap (HP):    -0.28
Credit gap (HP): 1.08


### **Conclusion: Current Cycle Position and External Validation**

The cycle clock diagnostics place the U.S. economy in a **late-cycle slowdown** as of **2025Q3**.  
This regime is characterized by a **negative output growth gap** and a **positive credit growth gap**, indicating that real economic activity has decelerated below its long-run trend while credit conditions remain relatively accommodative.

Specifically, the GDP growth gap is moderately negative (−0.28), signaling weakening real momentum, whereas the credit growth gap remains clearly positive (+1.08), suggesting that credit expansion has not yet fully adjusted to the slowdown in activity. This configuration is typical of late-cycle phases in which leverage continues to build or remains elevated even as growth fades.

From a macro-financial perspective, such an environment is inherently fragile. When growth slows before credit conditions tighten, the economy becomes increasingly sensitive to adverse shocks and policy miscalibration. Historically, late-cycle slowdowns often precede either a transition toward **deleveraging** or a **soft-landing scenario**, depending on the evolution of monetary policy and financial conditions.

Importantly, this model-based assessment is broadly consistent with recent macroeconomic commentary and forecasts reported in the financial press and institutional outlooks. Current narratives emphasize **below-trend but still positive U.S. growth**, resilient credit and financial conditions, and a monetary policy stance that remains restrictive but is gradually shifting toward accommodation. In this sense, the empirical signals extracted from the data align well with prevailing external assessments, reinforcing the plausibility of the identified late-cycle regime.

Overall, the cycle clock complements the composite Credit Cycle Index by providing a discrete regime classification that corroborates the continuous signals derived from the PCA-based framework. Together, they offer a coherent, data-driven interpretation of the current macroeconomic and credit cycle without relying on explicit forecasting assumptions.